# Kinetic Rates Transformation Test

## What is tested

For a composite system of two subsystems, relaxation can be transformed into the target
(eigen) basis in two ways: as a full Liouville superoperator or as a classical rate
matrix. This notebook checks that `transform_kronecker_superoperator` and
`transform_kronecker_rate_matrix` agree with each other and with direct calculations.

## List of tests

Each test is repeated for every test case listed in the "Test cases" section.

1. `test_superoperator_equivalence`: transforming the jump operators and then building the dissipator gives the same superoperator as `transform_kronecker_superoperator` and as a direct transformation.
2. `test_kinetic_equivalence`: the population block extracted from the transformed superoperator equals the directly transformed rate matrix.
3. `test_population_conservation`: every column of the transformed rate matrix sums to zero.

## How to run

Run all cells. The last two cells call `run_tests_simple()` and `run_tests_difficult()`.
Each test prints one line when it passes and raises `AssertionError` when it fails.

In [1]:
import os
import sys

import torch

# Makes MaRs importable when the notebook is run from the repository without installation
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "..")))

from mars.population import transform as pt

COMPLEX_DTYPE = torch.complex128
DEVICE = torch.device("cpu")

## Helper functions

In [2]:
def random_unitary(dim: int) -> torch.Tensor:
    """Return a random unitary matrix."""
    matrix = torch.randn(dim, dim, dtype=COMPLEX_DTYPE, device=DEVICE)
    q, r = torch.linalg.qr(matrix)
    phase = torch.diagonal(r)
    phase = phase / phase.abs().clamp_min(1e-12)
    return q * phase.conj()


def random_rate_matrix(dim: int) -> torch.Tensor:
    """Return a random rate matrix W[target, source] with nonnegative rates and a zero diagonal."""
    rates = torch.rand(dim, dim, dtype=torch.float64, device=DEVICE)
    rates.fill_diagonal_(0.0)
    return rates


def setup_test_case(d1: int, d2: int, seed: int, batch_size: int = 1) -> dict:
    """Generate random bases, jump operators and rate matrices for two subsystems."""
    torch.manual_seed(seed)
    batch_shape = (batch_size,) if batch_size > 1 else ()

    def expand(tensor: torch.Tensor) -> torch.Tensor:
        """Repeat a tensor along the batch dimension when a batch is requested."""
        return tensor.expand(batch_shape + tensor.shape) if batch_size > 1 else tensor

    basis_1 = expand(random_unitary(d1))
    basis_2 = expand(random_unitary(d2))
    target_basis = expand(random_unitary(d1 * d2))

    coefficients = pt.compute_clebsch_gordan_coeffs(target_basis, [basis_1, basis_2])
    unitary = coefficients.reshape(*batch_shape, d1 * d2, d1 * d2)

    jump_1 = torch.randn(batch_shape + (d1, d1), dtype=COMPLEX_DTYPE, device=DEVICE)
    jump_2 = torch.randn(batch_shape + (d2, d2), dtype=COMPLEX_DTYPE, device=DEVICE)

    rates_1 = expand(random_rate_matrix(d1))
    rates_2 = expand(random_rate_matrix(d2))
    dissipator_1 = pt.Liouvilleator().lindblad_dissipator_from_rates(rates_1).to(COMPLEX_DTYPE)
    dissipator_2 = pt.Liouvilleator().lindblad_dissipator_from_rates(rates_2).to(COMPLEX_DTYPE)

    return {
        "coefficients": coefficients,
        "unitary": unitary,
        "jump_1": jump_1,
        "jump_2": jump_2,
        "rates_1": rates_1,
        "rates_2": rates_2,
        "dissipator_1": dissipator_1,
        "dissipator_2": dissipator_2,
        "d1": d1,
        "d2": d2,
    }


def transformed_kinetic_matrices(case: dict) -> tuple[torch.Tensor, torch.Tensor]:
    """Return the rate matrix obtained directly and the one extracted from the transformed superoperator."""
    superop = pt.transform_kronecker_superoperator(
        [case["dissipator_1"], case["dissipator_2"]],
        case["unitary"],
        apply_secular_approximation=False,
    )
    extracted = pt.extract_transition_matrix_from_superoperator(superop).real

    direct = pt.transform_kronecker_rate_matrix(
        [case["rates_1"], case["rates_2"]],
        pt.get_product_to_target_unitary(case["coefficients"], n=2),
    )
    # Rebuild the diagonal from the outgoing rates
    index = torch.arange(direct.shape[-1])
    direct[..., index, index] = 0.0
    direct[..., index, index] = -direct.sum(dim=-2)
    return direct, extracted

## Tests

In [3]:
def test_superoperator_equivalence(case: dict) -> None:
    """Check that the order of 'transform jumps' and 'build dissipator' does not change the superoperator."""
    jump_1, jump_2, unitary = case["jump_1"], case["jump_2"], case["unitary"]
    d1, d2 = case["d1"], case["d2"]
    batch_shape = jump_1.shape[:-2]

    identity_1 = torch.eye(d1, dtype=jump_1.dtype, device=jump_1.device).expand(batch_shape + (d1, d1))
    identity_2 = torch.eye(d2, dtype=jump_2.dtype, device=jump_2.device).expand(batch_shape + (d2, d2))
    jump_1_full = pt.batched_kron(jump_1, identity_2)
    jump_2_full = pt.batched_kron(identity_1, jump_2)
    dissipator = pt.Liouvilleator.lindblad_dissipator_from_operator

    # Path 1: transform the jump operators, then build the dissipator
    unitary_dagger = unitary.conj().transpose(-1, -2)
    from_jumps = dissipator(unitary @ jump_1_full @ unitary_dagger) + dissipator(unitary @ jump_2_full @ unitary_dagger)

    # Path 2: build the local dissipators, then transform them with MaRs
    from_mars = pt.transform_kronecker_superoperator(
        [dissipator(jump_1), dissipator(jump_2)], unitary, apply_secular_approximation=False
    )

    # Path 3: build the full dissipator, then apply the Liouville-space transformation directly
    liouville_unitary = pt.batched_kron(unitary, unitary.conj())
    from_direct = (
        liouville_unitary
        @ (dissipator(jump_1_full) + dissipator(jump_2_full))
        @ liouville_unitary.conj().transpose(-1, -2)
    )

    tolerance = 1e-10
    difference_mars = (from_jumps - from_mars).abs().max().item()
    difference_direct = (from_jumps - from_direct).abs().max().item()
    assert difference_mars < tolerance, f"MaRs transformation differs by {difference_mars:.2e}"
    assert difference_direct < tolerance, f"Direct transformation differs by {difference_direct:.2e}"


def test_kinetic_equivalence(case: dict) -> None:
    """Check that the population block of the transformed superoperator equals the transformed rate matrix."""
    direct, extracted = transformed_kinetic_matrices(case)
    absolute_difference = (direct - extracted).abs().max().item()
    relative_difference = (torch.norm(direct - extracted) / (torch.norm(extracted) + 1e-12)).item()
    assert absolute_difference < 1e-8, f"Absolute difference is {absolute_difference:.2e}"
    assert relative_difference < 1e-6, f"Relative difference is {relative_difference:.2e}"


def test_population_conservation(case: dict) -> None:
    """Check that every column of the transformed rate matrix sums to zero."""
    direct, _ = transformed_kinetic_matrices(case)
    max_column_sum = direct.sum(dim=-2).abs().max().item()
    assert max_column_sum < 1e-8, f"Largest column sum is {max_column_sum:.2e}"

## Test cases

In [4]:
SIMPLE_CASES = [
    {"d1": 2, "d2": 2, "seed": 0},
    {"d1": 3, "d2": 4, "seed": 42},
]

DIFFICULT_CASES = [
    {"d1": 4, "d2": 3, "seed": 123},
    {"d1": 5, "d2": 2, "seed": 99},
    {"d1": 3, "d2": 3, "seed": 777, "batch_size": 4},
    {"d1": 4, "d2": 5, "seed": 2024},
]


def run_cases(cases: list[dict]) -> None:
    """Run all tests for each test case and print one line per case."""
    for parameters in cases:
        case = setup_test_case(**parameters)
        test_superoperator_equivalence(case)
        test_kinetic_equivalence(case)
        test_population_conservation(case)
        label = ", ".join(f"{key}={value}" for key, value in parameters.items())
        print(f"Kinetic rates test passed ({label}: superoperator, rate matrix and conservation agree)")

## Run the tests

In [5]:
def run_tests_simple():
    """Run the small test cases."""
    run_cases(SIMPLE_CASES)
    print(f"\nAll {len(SIMPLE_CASES)} simple kinetic rates test cases passed.")


def run_tests_difficult():
    """Run the larger test cases and the batched test case."""
    run_cases(DIFFICULT_CASES)
    print(f"\nAll {len(DIFFICULT_CASES)} difficult kinetic rates test cases passed.")

In [6]:
run_tests_simple()

Kinetic rates test passed (d1=2, d2=2, seed=0: superoperator, rate matrix and conservation agree)
Kinetic rates test passed (d1=3, d2=4, seed=42: superoperator, rate matrix and conservation agree)

All 2 simple kinetic rates test cases passed.


In [7]:
run_tests_difficult()

Kinetic rates test passed (d1=4, d2=3, seed=123: superoperator, rate matrix and conservation agree)
Kinetic rates test passed (d1=5, d2=2, seed=99: superoperator, rate matrix and conservation agree)
Kinetic rates test passed (d1=3, d2=3, seed=777, batch_size=4: superoperator, rate matrix and conservation agree)
Kinetic rates test passed (d1=4, d2=5, seed=2024: superoperator, rate matrix and conservation agree)

All 4 difficult kinetic rates test cases passed.
